In [ ]:
# Install all libraries we need for this project
# transformers = load AI models from HuggingFace
# bitsandbytes = lets us compress the model to fit on free GPU
# accelerate = helps run the model efficiently on GPU
# pandas = read and write CSV files

!pip install transformers bitsandbytes accelerate pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from google.colab import files

# Upload dataset_clean.csv when the dialog appears
uploaded = files.upload()

# Read the CSV into a table called "questions"
questions = pd.read_csv("dataset_clean.csv")

# Check it loaded correctly
print(f"Total questions: {len(questions)}")
print(questions.head(3))

Saving dataset_clean.csv to dataset_clean.csv
Total questions: 643
             id                                             prompt
0  CORP-TAX-001  Was ist die steuerliche Bemessungsgrundlage fü...
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn ...
2  CORP-TAX-003  Welche Körperschaften sind verpflichtet, sämtl...


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Check if GPU is available (should say "cuda" on Colab T4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 4-bit quantization config
# This compresses the model so it fits on the free T4 GPU
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# Mistral 7B - good at German, follows instructions
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model in 4-bit... (2-3 minutes)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"  # automatically puts model on GPU
)

print("Model ready!")

Using device: cuda
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model ready!


In [ ]:
# Test on just the first question before running all 643
# This makes sure everything works correctly

question = questions["prompt"][0]
print("Question:", question)

# prompt - asks for short, precise answer with law references
# kurz und präzise = short and precise
# Nenne den relevanten Paragraphen = cite the relevant law paragraph
prompt = f"[INST] Du bist ein österreichischer Steuerexperte. Beantworte die folgende Frage kurz und präzise auf Deutsch. Nenne wenn möglich den relevanten Paragraphen (z.B. § 7 KStG):\n\n{question} [/INST]"

# Tokenize and move to GPU
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate the answer
output_tokens = model.generate(
    **inputs,
    max_new_tokens=200,   # max answer length
    do_sample=True,       # more natural answers
    temperature=0.3,      # low = more factual, less creative
    pad_token_id=tokenizer.eos_token_id
)

# Decode only the new answer part (skip the input prompt)
input_length = inputs["input_ids"].shape[1]
answer = tokenizer.decode(output_tokens[0][input_length:], skip_special_tokens=True)

print("\nAnswer:", answer)

Question: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?

Answer: Die Steuerbemessungsgrundlage für die Körperschaftsteuer in Österreich ist der Gewinn des Unternehmens nach dem Prinzip der Einkommensbesteuerung (§ 1 Abs. 1 KStG). Dieser Gewinn ergibt sich aus den im Geschäftsjahr anfallenden Umsätzen und den dazugehörigen Betriebsausgaben (§ 1 Abs. 2 KStG). Eventuelle Steuervorzüge oder -abzüge (z.B. nach § 23a KStG) müssen berücksichtigt werden.


In [ ]:
import os

# Output file name
output_file = "model1_mistral_results.csv"

# Check if we already have some results saved (from before disconnect)
if os.path.exists(output_file):
    existing_results = pd.read_csv(output_file)
    done_ids = set(existing_results["id"].tolist())
    all_answers = existing_results.to_dict("records")
    print(f"Resuming! Already done: {len(done_ids)} questions")
else:
    done_ids = set()
    all_answers = []
    print("Starting fresh!")

# Loop through all questions
for i, row in questions.iterrows():

    question_id = row["id"]
    question_text = row["prompt"]

    # Skip questions we already answered
    if question_id in done_ids:
        continue

    # Build prompt
    prompt = f"[INST] Du bist ein österreichischer Steuerexperte. Beantworte die folgende Frage kurz und präzise auf Deutsch. Nenne wenn möglich den relevanten Paragraphen (z.B. § 7 KStG):\n\n{question_text} [/INST]"

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt",
                      max_length=512, truncation=True).to(device)

    # Generate answer
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode answer
    input_length = inputs["input_ids"].shape[1]
    answer = tokenizer.decode(output_tokens[0][input_length:],
                             skip_special_tokens=True)

    # Store result
    all_answers.append({"id": question_id, "answer": answer})

    # Save to CSV after every single question
    # This way if Colab disconnects we don't lose progress
    pd.DataFrame(all_answers).to_csv(output_file, index=False)

    # Print progress every 50 questions
    if (i + 1) % 50 == 0:
        print(f"Progress: {len(all_answers)} / {len(questions)}")

print("All done!")
print(pd.read_csv(output_file).head(3))

Starting fresh!
Progress: 50 / 643
Progress: 100 / 643
Progress: 150 / 643
Progress: 200 / 643
Progress: 250 / 643
Progress: 300 / 643
Progress: 350 / 643
Progress: 400 / 643
Progress: 450 / 643
Progress: 500 / 643
Progress: 550 / 643
Progress: 600 / 643
All done!
             id                                             answer
0  CORP-TAX-001  Die Steuerbemessungsgrundlage für die Körpersc...
1  CORP-TAX-002  Die Zinsenlose Geldvergabe an den Gesellschaft...
2  CORP-TAX-003  Körperschaften, die Gewerbebetrieb betreiben u...


In [ ]:
# Download the results CSV to your computer
from google.colab import files
files.download("model1_mistral_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>